<a href="https://colab.research.google.com/github/sajjkavinda/ML-based-Intrution-detection-system/blob/main/Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Cleaning

This notebook performs the data cleaning stage of the proposed IDS.

The preprocessing pipeline consists of:

1. Creating a reproducible subset of the dataset
2. Removing duplicate records
3. Handling missing and infinite values
4. Converting data types
5. Removing unnecessary features
6. Encoding the target variable
7. Saving the cleaned dataset for feature engineering

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Import libraries

In [16]:
import pandas as pd
import numpy as np

from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

Set dataset and output paths

In [17]:
PROJECT_DIR = Path("/content/drive/MyDrive/MSc_Dissertation")

DATASET_DIR = PROJECT_DIR / "Dataset"

PROCESSED_DIR = DATASET_DIR / "Processed"

PROCESSED_DIR.mkdir(exist_ok=True)

DATA_FILE = DATASET_DIR / "Friday-16-02-2018_TrafficForML_CICFlowMeter.csv"

Load the dataset

In [18]:
df = pd.read_csv(DATA_FILE, low_memory=False)

print(df.shape)

(1048575, 80)


Create a subset from the dataset

Subset size = 300000 rows

In [19]:
subset_size = 300000

df = (
    df.groupby("Label", group_keys=False)
      .apply(lambda x: x.sample(
          n=max(1, int(len(x)/len(df)*subset_size)),
          random_state=42))
      .reset_index(drop=True)
)

print(df.shape)

(299999, 80)


Handle infinite values

In [21]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

Handle missing values

In [22]:
print(df.isnull().sum().sum())

df.dropna(inplace=True)

print(df.shape)

0
(269412, 80)


Remove the column Timestamp since it's not required for IDS

In [23]:
df.drop(columns=["Timestamp"], inplace=True)

Records are in objects format. Convert them to numeric format except labels

In [24]:
for col in df.columns:

    if col != "Label":

        df[col] = pd.to_numeric(df[col], errors="coerce")

Remove missing values

In [25]:
df.dropna(inplace=True)

print(df.shape)

(269411, 79)


Encode labels

In [26]:
df["Label"].value_counts()

,count
Label,
DoS attacks-Hulk,129602
Benign,127811
DoS attacks-SlowHTTPTest,11998


In [27]:
# Remove leading/trailing spaces
df["Label"] = df["Label"].astype(str).str.strip()

# Encode labels
df["Label"] = df["Label"].map({
    "Benign": 0,
    "DoS attacks-Hulk": 1,
    "DoS attacks-SlowHTTPTest": 1
})

print(df["Label"].value_counts())

Label
1    141600
0    127811
Name: count, dtype: int64


Remove duplicate records

In [37]:
duplicate_count = df.duplicated().sum()

print(f"Duplicates before removal: {duplicate_count}")

df = df.drop_duplicates()

print(f"Shape after duplicate removal: {df.shape}")

Duplicates before removal: 77954
Shape after duplicate removal: (191457, 79)


In [39]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())
print("Shape:", df.shape)
df["Label"].value_counts()

Missing values: 0
Duplicates: 0
Shape: (191457, 79)


,count
Label,
0,127810
1,63647


Save the cleaned subset

In [40]:
from pathlib import Path

OUTPUT_FILE = Path(
    "/content/drive/MyDrive/MSc_Dissertation/Dataset/Processed/cleaned_dataset.csv"
)

df.to_csv(OUTPUT_FILE, index=False)

print("Saved successfully:")
print(OUTPUT_FILE)

Saved successfully:
/content/drive/MyDrive/MSc_Dissertation/Dataset/Processed/cleaned_dataset.csv
